# Thesis-Grade Probabilistic Forecast Evaluation

This notebook evaluates quantile forecasts for battery-trading model selection.
It is forecast-side only (no simulation/backtest execution).


## Section 0 — Setup and configuration

In [1]:
from __future__ import annotations

import json
import math
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)


In [2]:
# --- Notebook-level config ---
EVALUATION_MODE = "thesis"
PLOT_ENABLED = False
MAX_DISPLAY_ROWS = 100
JOIN_COVERAGE_THRESHOLD = 0.999


def _discover_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for cand in [cur] + list(cur.parents):
        if (cand / "src").exists() and (cand / "notebooks").exists():
            return cand
        if (cand / "pyproject.toml").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError(
        f"Could not discover repository root from start path: {start}. "
        "Run notebook from repo root or set REPO_ROOT explicitly."
    )


REPO_ROOT = _discover_repo_root(Path.cwd())

ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "benchmark"
FIG_ROOT = ARTIFACT_ROOT / "figures"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
if PLOT_ENABLED:
    FIG_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_MANIFESTS = {
    "xgb": REPO_ROOT / "artifacts" / "model_runs" / "latest_xgboost.json",
    "tft": REPO_ROOT / "artifacts" / "model_runs" / "latest_tft.json",
    "linear_torch": REPO_ROOT / "artifacts" / "model_runs" / "latest_linear_torch.json",
}

PREDICTION_TABLES: dict[str, Path] = {
    # "xgb": REPO_ROOT / "artifacts" / "..." / "xgb_predictions.parquet",
    # "tft": REPO_ROOT / "artifacts" / "..." / "tft_predictions.parquet",
    # "linear_torch": REPO_ROOT / "artifacts" / "..." / "linear_predictions.parquet",
}

MODEL_IDS = ["xgb", "tft", "linear_torch"]
TARGETS = [
    "da_price",
    "afrr_capacity_price_pos",
    "afrr_capacity_price_neg",
    "afrr_activation_price_pos",
    "afrr_activation_price_neg",
    "afrr_activation_rate_pos",
    "afrr_activation_rate_neg",
]

PRED_KEY_TO_TARGET = {
    "pred_da_price": "da_price",
    "pred_afrr_capacity_price_pos": "afrr_capacity_price_pos",
    "pred_afrr_capacity_price_neg": "afrr_capacity_price_neg",
    "pred_afrr_activation_price_pos": "afrr_activation_price_pos",
    "pred_afrr_activation_price_neg": "afrr_activation_price_neg",
    "pred_afrr_activation_rate_pos": "afrr_activation_rate_pos",
    "pred_afrr_activation_rate_neg": "afrr_activation_rate_neg",
}

TRUTH_COLUMN_CANDIDATES = {
    "pred_da_price": [
        "da_price",
        "target_da_price",
        "true_da_price",
    ],
    "pred_afrr_capacity_price_pos": [
        "afrr_capacity_price_pos",
        "target_afrr_capacity_price_pos",
        "true_afrr_capacity_price_pos",
    ],
    "pred_afrr_capacity_price_neg": [
        "afrr_capacity_price_neg",
        "target_afrr_capacity_price_neg",
        "true_afrr_capacity_price_neg",
    ],
    "pred_afrr_activation_price_pos": [
        "afrr_activation_price_vwap_pos",
        "afrr_activation_price_pos",
        "target_afrr_activation_price_pos",
        "true_afrr_activation_price_pos",
    ],
    "pred_afrr_activation_price_neg": [
        "afrr_activation_price_vwap_neg",
        "afrr_activation_price_neg",
        "target_afrr_activation_price_neg",
        "true_afrr_activation_price_neg",
    ],
    "pred_afrr_activation_rate_pos": [
        "activation_rate_phys_pos",
        "afrr_activation_rate_pos",
        "target_afrr_activation_rate_pos",
        "true_afrr_activation_rate_pos",
    ],
    "pred_afrr_activation_rate_neg": [
        "activation_rate_phys_neg",
        "afrr_activation_rate_neg",
        "target_afrr_activation_rate_neg",
        "true_afrr_activation_rate_neg",
    ],
}

QUANTILES_REQUIRED = ["p10", "p30", "p50", "p70", "p90"]
KEY_QUANTILE_PAIRS = ["p50_p50", "p70_p90"]

QUANTILE_PAIR_MAPPING = pd.DataFrame(
    [
        {"scenario": "p50_p50", "target": t, "selected_quantile": "p50", "risk_note": "central"}
        for t in TARGETS
    ]
    + [
        {
            "scenario": "p70_p90",
            "target": t,
            "selected_quantile": ("p90" if "activation" in t else "p70"),
            "risk_note": "aggressive_tail_seeking",
        }
        for t in TARGETS
    ]
)
QUANTILE_PAIR_MAPPING.to_csv(ARTIFACT_ROOT / "quantile_pair_mapping.csv", index=False)

print("REPO_ROOT:", REPO_ROOT)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


REPO_ROOT: /Users/leori/Code/energyTrading
ARTIFACT_ROOT: /Users/leori/Code/energyTrading/artifacts/benchmark


In [3]:
def _read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {path}")


def _resolve_latest_manifest_pointer(latest_path: Path) -> Path:
    if not latest_path.exists():
        raise FileNotFoundError(f"Manifest missing: {latest_path}")
    data = json.loads(latest_path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Latest manifest must be dict: {latest_path}")

    run_manifest_rel = data.get("manifest_path") or data.get("manifest_path_abs")
    if isinstance(run_manifest_rel, str) and run_manifest_rel.strip():
        cand = (latest_path.parent / run_manifest_rel).resolve()
        if cand.exists():
            return cand

    run_id = data.get("run_id")
    if isinstance(run_id, str) and run_id.strip():
        cand = (latest_path.parent / run_id / "manifest.json").resolve()
        if cand.exists():
            return cand

    if any(k in data for k in ("predictions_path", "prediction_path", "outputs")):
        return latest_path

    raise KeyError(
        f"Could not resolve run manifest from latest manifest pointer: {latest_path}. "
        f"Available keys: {list(data.keys())}"
    )


def _resolve_truth_path_from_manifest(run_manifest_path: Path, run_manifest: dict) -> Path:
    gt = run_manifest.get("ground_truth", {}) if isinstance(run_manifest, dict) else {}
    p = gt.get("default_path") if isinstance(gt, dict) else None
    if not isinstance(p, str) or not p.strip():
        raise KeyError(
            f"Missing ground_truth.default_path in run manifest: {run_manifest_path}"
        )
    cand = (run_manifest_path.parent / p).resolve()
    if not cand.exists():
        raise FileNotFoundError(f"Ground truth file not found from manifest: {cand}")
    return cand


def _resolve_bundle_split_truth_path(repo_root: Path, bundle_name: str, split: str) -> Path | None:
    cand = (repo_root / "data" / "model_input" / str(bundle_name) / f"{split}.parquet").resolve()
    return cand if cand.exists() else None


def _resolve_target_pred_key(target: str) -> str:
    for k, v in PRED_KEY_TO_TARGET.items():
        if v == target:
            return k
    raise KeyError(f"No prediction-key mapping for target={target}")


def _resolve_truth_column_for_pred_key(truth_df: pd.DataFrame, pred_key: str) -> str:
    if pred_key not in TRUTH_COLUMN_CANDIDATES:
        raise KeyError(f"No truth-column candidates configured for pred_key={pred_key}")
    candidates = TRUTH_COLUMN_CANDIDATES[pred_key]
    available = list(truth_df.columns)
    matches = [c for c in candidates if c in truth_df.columns]
    if len(matches) == 1:
        return matches[0]
    if len(matches) == 0:
        nearby = sorted([c for c in available if any(tok in c.lower() for tok in pred_key.split("_") if tok)])[:25]
        raise KeyError(
            f"No truth column match for pred_key={pred_key}. Tried={candidates}. Nearby columns={nearby}"
        )
    raise ValueError(
        f"Ambiguous truth mapping for pred_key={pred_key}. Matches={matches}. Candidates={candidates}"
    )


def _iter_bundle_long_tables(run_manifest_path: Path, bundle_obj: dict) -> list[tuple[str, str, Path]]:
    out = []
    pl = bundle_obj.get("predictions_long") if isinstance(bundle_obj, dict) else None
    if not isinstance(pl, dict):
        return out
    for split, split_obj in pl.items():
        if not isinstance(split_obj, dict):
            continue
        for pred_key, rel_path in split_obj.items():
            if not isinstance(rel_path, str) or not rel_path.strip():
                continue
            out.append((str(split), str(pred_key), (run_manifest_path.parent / rel_path).resolve()))
    return out


def _load_model_from_run_manifest(model_id: str, latest_manifest_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    run_manifest_path = _resolve_latest_manifest_pointer(latest_manifest_path)
    run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    if not isinstance(run_manifest, dict):
        raise ValueError(f"Run manifest must be dict: {run_manifest_path}")

    bundles = run_manifest.get("bundles")
    if not isinstance(bundles, dict):
        raise KeyError(f"Missing bundles in run manifest: {run_manifest_path}")

    truth_default_path = _resolve_truth_path_from_manifest(run_manifest_path, run_manifest)
    rows = []
    audits = []

    for bundle_name, bundle_obj in bundles.items():
        if not isinstance(bundle_obj, dict):
            continue

        long_tables = _iter_bundle_long_tables(run_manifest_path, bundle_obj)
        for split, pred_key, pred_path in long_tables:
            if pred_key not in PRED_KEY_TO_TARGET:
                continue
            target = PRED_KEY_TO_TARGET[pred_key]

            if not pred_path.exists():
                raise FileNotFoundError(f"Predictions_long file missing: {pred_path}")
            pred = _read_table(pred_path)

            req = ["snapshot_time_utc", "target_time_utc"] + QUANTILES_REQUIRED
            miss = [c for c in req if c not in pred.columns]
            if miss:
                raise KeyError(f"{model_id}/{target}/{split}: missing required columns in {pred_path.name}: {miss}")

            truth_path = _resolve_bundle_split_truth_path(REPO_ROOT, str(bundle_name), split)
            if truth_path is None:
                truth_path = truth_default_path
            truth = _read_table(truth_path)

            ts_truth = "timestamp_utc" if "timestamp_utc" in truth.columns else ("target_time_utc" if "target_time_utc" in truth.columns else None)
            if ts_truth is None:
                raise KeyError(f"Truth data must contain timestamp_utc or target_time_utc: {truth_path}")

            truth = truth.copy()
            truth["timestamp_utc"] = pd.to_datetime(truth[ts_truth], utc=True, errors="coerce")
            if truth["timestamp_utc"].isna().any():
                raise ValueError(f"{model_id}/{target}/{split}: invalid truth timestamps in {truth_path}")

            truth_col = _resolve_truth_column_for_pred_key(truth, pred_key)
            truth_sub = truth[["timestamp_utc", truth_col]].copy().rename(columns={truth_col: "y_true"})
            truth_sub = truth_sub.drop_duplicates(subset=["timestamp_utc"], keep="last")

            pred_sub = pred.copy()
            pred_sub["snapshot_time_utc"] = pd.to_datetime(pred_sub["snapshot_time_utc"], utc=True, errors="coerce")
            pred_sub["target_time_utc"] = pd.to_datetime(pred_sub["target_time_utc"], utc=True, errors="coerce")
            bad_ts = pred_sub["snapshot_time_utc"].isna() | pred_sub["target_time_utc"].isna()
            if bad_ts.any():
                raise ValueError(f"{model_id}/{target}/{split}: invalid timestamps in {pred_path.name}: {int(bad_ts.sum())}")

            out = pred_sub.merge(
                truth_sub,
                left_on="target_time_utc",
                right_on="timestamp_utc",
                how="left",
                validate="m:1",
            )
            pred_rows = int(len(pred_sub))
            join_rows = int(len(out))
            missing_truth_rows = int(out["y_true"].isna().sum())
            coverage = 0.0 if pred_rows == 0 else float((pred_rows - missing_truth_rows) / pred_rows)

            status = "ok"
            if coverage < JOIN_COVERAGE_THRESHOLD:
                status = "coverage_fail"

            audits.append(
                {
                    "model": model_id,
                    "target": target,
                    "split": split,
                    "prediction_file": str(pred_path),
                    "truth_file": str(truth_path),
                    "resolved_truth_col": truth_col,
                    "pred_rows": pred_rows,
                    "truth_rows": int(len(truth_sub)),
                    "join_rows": join_rows,
                    "join_coverage_pct": coverage * 100.0,
                    "missing_truth_rows": missing_truth_rows,
                    "min_snapshot_time_utc": pred_sub["snapshot_time_utc"].min(),
                    "max_snapshot_time_utc": pred_sub["snapshot_time_utc"].max(),
                    "min_target_time_utc": pred_sub["target_time_utc"].min(),
                    "max_target_time_utc": pred_sub["target_time_utc"].max(),
                    "min_truth_timestamp_utc": truth_sub["timestamp_utc"].min(),
                    "max_truth_timestamp_utc": truth_sub["timestamp_utc"].max(),
                    "available_quantiles": ",".join([q for q in QUANTILES_REQUIRED if q in pred_sub.columns]),
                    "status": status,
                }
            )

            if coverage < JOIN_COVERAGE_THRESHOLD:
                missing_examples = (
                    out.loc[out["y_true"].isna(), "target_time_utc"]
                    .dropna()
                    .astype(str)
                    .head(10)
                    .tolist()
                )
                raise ValueError(
                    f"{model_id}/{target}/{split}: truth join coverage {coverage:.4%} below threshold "
                    f"{JOIN_COVERAGE_THRESHOLD:.4%}. Missing rows={missing_truth_rows}/{pred_rows}. "
                    f"Missing target_time_utc examples={missing_examples}"
                )

            out2 = pd.DataFrame(
                {
                    "model": model_id,
                    "target": target,
                    "split": split,
                    "target_time_utc": out["target_time_utc"],
                    "snapshot_time_utc": out["snapshot_time_utc"],
                    "lead_time_h": pd.to_numeric(out.get("lead_time_h"), errors="coerce"),
                    "y_true": pd.to_numeric(out["y_true"], errors="coerce"),
                }
            )
            if out2["lead_time_h"].isna().all():
                out2["lead_time_h"] = (
                    (out2["target_time_utc"] - out2["snapshot_time_utc"]).dt.total_seconds() / 3600.0
                )

            for q in QUANTILES_REQUIRED:
                out2[q] = pd.to_numeric(out[q], errors="coerce")

            if out2[QUANTILES_REQUIRED].isna().any().any():
                bad_cols = out2[QUANTILES_REQUIRED].isna().sum()
                bad_cols = bad_cols[bad_cols > 0]
                raise ValueError(f"{model_id}/{target}/{split}: NaN quantiles in {pred_path.name}: {bad_cols.to_dict()}")

            out2["source_path"] = str(pred_path)
            rows.append(out2)

    if not rows:
        raise RuntimeError(
            f"No usable predictions_long tables found for model {model_id} from manifest {latest_manifest_path}"
        )

    norm = pd.concat(rows, ignore_index=True)
    req_cols = ["model", "target", "split", "target_time_utc", "snapshot_time_utc", "lead_time_h", "y_true"] + QUANTILES_REQUIRED
    missing = [c for c in req_cols if c not in norm.columns]
    if missing:
        raise KeyError(f"{model_id}: normalized table missing required columns: {missing}")
    return norm, pd.DataFrame(audits)


normalized_frames = []
audit_frames = []
for model_id in MODEL_IDS:
    if model_id in PREDICTION_TABLES:
        raise NotImplementedError(
            "Direct PREDICTION_TABLES override currently expects already-normalized data; "
            "for this notebook use manifest-based loading or extend loader explicitly."
        )
    norm, audit = _load_model_from_run_manifest(model_id, MODEL_MANIFESTS[model_id])
    normalized_frames.append(norm)
    audit_frames.append(audit)

all_preds = pd.concat(normalized_frames, ignore_index=True)
truth_mapping_audit = pd.concat(audit_frames, ignore_index=True)
truth_mapping_audit = truth_mapping_audit.sort_values(["model", "target", "split"]).reset_index(drop=True)
truth_mapping_audit.to_csv(ARTIFACT_ROOT / "benchmark_truth_mapping_audit.csv", index=False)
print(truth_mapping_audit.head(min(MAX_DISPLAY_ROWS, 10)))

if all_preds.empty:
    raise RuntimeError("No normalized prediction data loaded.")

all_preds = all_preds.sort_values(["model", "target", "split", "target_time_utc", "snapshot_time_utc"]).reset_index(drop=True)
print(all_preds.head(min(MAX_DISPLAY_ROWS, 5)))
print("rows:", len(all_preds))


ValueError: xgb/da_price/val: truth join coverage 99.4360% below threshold 99.9000%. Missing rows=1176/208512. Missing target_time_utc examples=['2025-01-01 00:00:00+00:00', '2025-01-01 00:00:00+00:00', '2025-01-01 01:00:00+00:00', '2025-01-01 00:00:00+00:00', '2025-01-01 01:00:00+00:00', '2025-01-01 02:00:00+00:00', '2025-01-01 00:00:00+00:00', '2025-01-01 01:00:00+00:00', '2025-01-01 02:00:00+00:00', '2025-01-01 03:00:00+00:00']

## Section 1 — Data inventory and schema audit

In [ ]:
def _crossing_stats(df: pd.DataFrame) -> tuple[float, float]:
    q = df[QUANTILES_REQUIRED].copy()
    diffs = q.diff(axis=1).iloc[:, 1:]
    bad = (diffs < 0).any(axis=1)
    rate = float(bad.mean()) if len(bad) else float("nan")
    max_v = float((-diffs.clip(upper=0.0)).max().max()) if len(diffs) else 0.0
    return rate, max_v

inv_rows = []
for (target, model, split), g in all_preds.groupby(["target", "model", "split"], dropna=False):
    q_available = [q for q in QUANTILES_REQUIRED if q in g.columns]
    missing_q = [q for q in QUANTILES_REQUIRED if q not in q_available]
    dup_cnt = int(g.duplicated(subset=["snapshot_time_utc", "target_time_utc", "target", "model", "split"]).sum())
    crossing_rate, crossing_max = _crossing_stats(g)
    inv_rows.append({
        "target": target,
        "model": model,
        "split": split,
        "n_rows": int(len(g)),
        "available_quantiles": ",".join(q_available),
        "missing_required_quantiles": ",".join(missing_q),
        "target_time_min_utc": g["target_time_utc"].min(),
        "target_time_max_utc": g["target_time_utc"].max(),
        "snapshot_time_min_utc": g["snapshot_time_utc"].min(),
        "snapshot_time_max_utc": g["snapshot_time_utc"].max(),
        "lead_time_min_h": float(pd.to_numeric(g["lead_time_h"], errors="coerce").min()),
        "lead_time_max_h": float(pd.to_numeric(g["lead_time_h"], errors="coerce").max()),
        "duplicate_rows_snapshot_target_lead": dup_cnt,
        "quantile_crossing_rate_before_repair": crossing_rate,
        "max_crossing_violation_before_repair": crossing_max,
    })

inventory = pd.DataFrame(inv_rows).sort_values(["target", "model", "split"]).reset_index(drop=True)
inventory.to_csv(ARTIFACT_ROOT / "benchmark_data_inventory.csv", index=False)
display(inventory.head(MAX_DISPLAY_ROWS))


## Section 2 — Core probabilistic forecast metrics

In [ ]:
def _pinball(y: np.ndarray, qhat: np.ndarray, q: float) -> np.ndarray:
    e = y - qhat
    return np.where(e >= 0.0, q * e, (q - 1.0) * e)


def _crps_approx_from_quantiles(y: np.ndarray, pred: pd.DataFrame) -> np.ndarray:
    q_levels = np.array([int(q[1:]) / 100.0 for q in QUANTILES_REQUIRED], dtype=float)
    vals = np.vstack([_pinball(y, pred[q].to_numpy(dtype=float), ql) for q, ql in zip(QUANTILES_REQUIRED, q_levels)])
    return 2.0 * np.nanmean(vals, axis=0)


def _winkler(y: np.ndarray, lo: np.ndarray, hi: np.ndarray, alpha: float) -> np.ndarray:
    width = hi - lo
    s = width.copy()
    below = y < lo
    above = y > hi
    s[below] = width[below] + (2.0 / alpha) * (lo[below] - y[below])
    s[above] = width[above] + (2.0 / alpha) * (y[above] - hi[above])
    return s


def _safe_wmape(y: np.ndarray, yhat: np.ndarray) -> float:
    den = np.sum(np.abs(y))
    if den <= 1e-12:
        return float("nan")
    return float(np.sum(np.abs(y - yhat)) / den)


metric_rows = []
for (target, model, split), g in all_preds.groupby(["target", "model", "split"], dropna=False):
    gg = g.dropna(subset=["y_true"] + QUANTILES_REQUIRED).copy()
    if gg.empty:
        continue
    y = gg["y_true"].to_numpy(dtype=float)
    p50 = gg["p50"].to_numpy(dtype=float)

    pin = {q: float(np.mean(_pinball(y, gg[q].to_numpy(dtype=float), int(q[1:]) / 100.0))) for q in QUANTILES_REQUIRED}
    crps = _crps_approx_from_quantiles(y, gg)

    cov_10_90 = float(((y >= gg["p10"].to_numpy(dtype=float)) & (y <= gg["p90"].to_numpy(dtype=float))).mean())
    cov_30_70 = float(((y >= gg["p30"].to_numpy(dtype=float)) & (y <= gg["p70"].to_numpy(dtype=float))).mean())
    wink_10_90 = float(np.mean(_winkler(y, gg["p10"].to_numpy(dtype=float), gg["p90"].to_numpy(dtype=float), alpha=0.2)))
    wink_30_70 = float(np.mean(_winkler(y, gg["p30"].to_numpy(dtype=float), gg["p70"].to_numpy(dtype=float), alpha=0.4)))

    crossing_rate, crossing_max = _crossing_stats(gg)

    # Directional accuracy on first differences where meaningful.
    y_diff = np.sign(np.diff(y))
    p_diff = np.sign(np.diff(p50))
    directional_acc = float(np.mean(y_diff == p_diff)) if len(y_diff) else float("nan")

    metric_rows.append({
        "target": target,
        "model": model,
        "split": split,
        "n": int(len(gg)),
        "mae_p50": float(np.mean(np.abs(y - p50))),
        "rmse_p50": float(np.sqrt(np.mean((y - p50) ** 2))),
        "bias_p50": float(np.mean(p50 - y)),
        "wmape_p50": _safe_wmape(y, p50),
        "directional_accuracy_p50": directional_acc,
        **{f"pinball_{q}": v for q, v in pin.items()},
        "mean_pinball": float(np.mean(list(pin.values()))),
        "crps_approx": float(np.mean(crps)),
        "coverage_p10_p90": cov_10_90,
        "coverage_p30_p70": cov_30_70,
        "winkler_p10_p90": wink_10_90,
        "winkler_p30_p70": wink_30_70,
        "crossing_rate_before_repair": crossing_rate,
        "max_crossing_violation_before_repair": crossing_max,
    })

forecast_metrics = pd.DataFrame(metric_rows).sort_values(["target", "model", "split"]).reset_index(drop=True)
forecast_metrics.to_csv(ARTIFACT_ROOT / "forecast_metrics_probabilistic.csv", index=False)
display(forecast_metrics.head(MAX_DISPLAY_ROWS))


## Section 3 — Gate-time / trading-time synced evaluation

In [ ]:
def _gate_subset(df: pd.DataFrame, target: str) -> pd.DataFrame:
    # Explicit convention. Adjust if your production gate semantics differ.
    ts = df["snapshot_time_utc"].dt.tz_convert("Europe/Berlin")
    lead = pd.to_numeric(df["lead_time_h"], errors="coerce")

    if target == "da_price":
        # Day-ahead decision-time subset proxy.
        mask = (ts.dt.hour == 12) & (lead >= 12) & (lead <= 36)
    elif "capacity" in target:
        # aFRR BCM gate subset proxy.
        mask = (ts.dt.hour == 8) & (lead >= 1) & (lead <= 24)
    else:
        # Activation/BEM relevant short-horizon subset.
        mask = (lead >= 0) & (lead <= 6)

    sub = df.loc[mask].copy()
    if sub.empty:
        warnings.warn(f"Gate-time subset is empty for target={target}. Check gate convention mapping.")
    return sub


gate_rows = []
for (target, model, split), g in all_preds.groupby(["target", "model", "split"], dropna=False):
    sub = _gate_subset(g, target).dropna(subset=["y_true"] + QUANTILES_REQUIRED)
    if sub.empty:
        continue

    y = sub["y_true"].to_numpy(dtype=float)
    p50 = sub["p50"].to_numpy(dtype=float)
    gate_rows.append({
        "target": target,
        "model": model,
        "split": split,
        "n_obs": int(len(sub)),
        "mae_p50": float(np.mean(np.abs(y - p50))),
        "rmse_p50": float(np.sqrt(np.mean((y - p50) ** 2))),
        "bias_p50": float(np.mean(p50 - y)),
        "pinball_p50": float(np.mean(_pinball(y, p50, 0.5))),
        "pinball_p70": float(np.mean(_pinball(y, sub["p70"].to_numpy(dtype=float), 0.7))),
        "pinball_p90": float(np.mean(_pinball(y, sub["p90"].to_numpy(dtype=float), 0.9))),
        "coverage_p10_p90": float(((y >= sub["p10"].to_numpy(dtype=float)) & (y <= sub["p90"].to_numpy(dtype=float))).mean()),
        "coverage_p70_p90": float(((y >= sub["p70"].to_numpy(dtype=float)) & (y <= sub["p90"].to_numpy(dtype=float))).mean()),
    })

gate_metrics = pd.DataFrame(gate_rows).sort_values(["target", "model", "split"]).reset_index(drop=True)
gate_metrics.to_csv(ARTIFACT_ROOT / "gate_time_forecast_metrics.csv", index=False)
display(gate_metrics.head(MAX_DISPLAY_ROWS))


## Section 4 — Tail performance for value-relevant events

In [ ]:
def _tail_masks(y: pd.Series, target: str) -> dict[str, pd.Series]:
    q80, q90, q95 = y.quantile([0.8, 0.9, 0.95]).tolist()
    q20, q10, q05 = y.quantile([0.2, 0.1, 0.05]).tolist()

    masks = {
        "top_20": y >= q80,
        "top_10": y >= q90,
        "top_05": y >= q95,
        "bottom_20": y <= q20,
        "bottom_10": y <= q10,
        "bottom_05": y <= q05,
    }

    if target == "afrr_activation_price_neg":
        # Economically extreme negative side: more negative values.
        masks["econ_extreme"] = y <= q10
    elif "rate" in target:
        masks["econ_extreme"] = y >= q90
    elif "capacity" in target or target == "afrr_activation_price_pos":
        masks["econ_extreme"] = y >= q90
    else:  # DA
        masks["econ_extreme"] = (y >= q90) | (y <= q10)
    return masks


def _event_classification(y: np.ndarray, score: np.ndarray, threshold: float) -> tuple[float, float, float]:
    pred = score >= threshold
    tp = float(np.sum(pred & y))
    fp = float(np.sum(pred & ~y))
    fn = float(np.sum(~pred & y))
    prec = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    rec = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = (2 * prec * rec / (prec + rec)) if np.isfinite(prec) and np.isfinite(rec) and (prec + rec) > 0 else float("nan")
    return prec, rec, f1


tail_rows = []
for (target, model, split), g in all_preds.groupby(["target", "model", "split"], dropna=False):
    gg = g.dropna(subset=["y_true", "p50", "p90"]).copy()
    if gg.empty:
        continue
    y = gg["y_true"]
    base_mae = float(np.mean(np.abs(gg["y_true"] - gg["p50"])))
    masks = _tail_masks(y, target)

    for mname, m in masks.items():
        sub = gg.loc[m]
        if sub.empty:
            continue
        e = sub["p50"] - sub["y_true"]
        tail_mae = float(np.mean(np.abs(e)))
        tail_rmse = float(np.sqrt(np.mean(e**2)))
        tail_bias = float(np.mean(e))
        ratio = float(tail_mae / base_mae) if base_mae > 1e-12 else float("nan")

        # Event capture using p90 as score surrogate.
        y_evt = m.to_numpy(dtype=bool)
        score = (gg["p90"] - gg["p50"]).to_numpy(dtype=float)
        thr = float(np.nanquantile(score, 0.9)) if len(score) else float("nan")
        prec, rec, f1 = _event_classification(y_evt, score, thr)

        # Value-weighted absolute error.
        vw = np.abs(sub["y_true"].to_numpy(dtype=float))
        vw_err = np.abs(e.to_numpy(dtype=float))
        vw_mae = float(np.sum(vw * vw_err) / np.sum(vw)) if float(np.sum(vw)) > 1e-12 else float("nan")

        tail_rows.append({
            "target": target,
            "model": model,
            "split": split,
            "tail_bucket": mname,
            "n": int(len(sub)),
            "tail_mae": tail_mae,
            "tail_rmse": tail_rmse,
            "tail_bias": tail_bias,
            "tail_mae_over_normal_mae": ratio,
            "spike_capture_rate": rec,
            "high_event_recall": rec,
            "high_event_precision": prec,
            "high_event_f1": f1,
            "avg_predicted_quantile_during_true_extremes": float(sub["p90"].mean()),
            "value_weighted_absolute_error": vw_mae,
        })

tail_df = pd.DataFrame(tail_rows).sort_values(["target", "model", "split", "tail_bucket"]).reset_index(drop=True)
tail_df.to_csv(ARTIFACT_ROOT / "tail_performance_value_events.csv", index=False)
display(tail_df.head(MAX_DISPLAY_ROWS))


## Section 5 — Joint value-event diagnostics

In [ ]:
def _joint_event_diag(g: pd.DataFrame, target: str) -> dict[str, float]:
    y = g["y_true"]
    p70 = g["p70"]
    p90 = g["p90"]

    y_hi = y >= y.quantile(0.9)
    p_hi = p90 >= p90.quantile(0.9)
    spread_hi = (p90 - p70) >= (p90 - p70).quantile(0.9)

    joint_true = y_hi & spread_hi
    joint_pred = p_hi & spread_hi

    tp = float((joint_true & joint_pred).sum())
    fp = float((~joint_true & joint_pred).sum())
    fn = float((joint_true & ~joint_pred).sum())

    prec = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    rec = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = (2 * prec * rec / (prec + rec)) if np.isfinite(prec) and np.isfinite(rec) and (prec + rec) > 0 else float("nan")

    captured_val = float(y.loc[joint_true & joint_pred].mean()) if (joint_true & joint_pred).any() else float("nan")
    missed_val = float(y.loc[joint_true & ~joint_pred].mean()) if (joint_true & ~joint_pred).any() else float("nan")

    return {
        "joint_event_recall": rec,
        "joint_event_precision": prec,
        "joint_event_f1": f1,
        "mean_realized_value_captured": captured_val,
        "mean_realized_value_missed": missed_val,
        "missed_high_value_event_count": float((joint_true & ~joint_pred).sum()),
    }


joint_rows = []
for (target, model, split), g in all_preds.groupby(["target", "model", "split"], dropna=False):
    gg = g.dropna(subset=["y_true", "p70", "p90"]).copy()
    if gg.empty:
        continue
    diag = _joint_event_diag(gg, target)
    joint_rows.append({"target": target, "model": model, "split": split, **diag})

joint_df = pd.DataFrame(joint_rows).sort_values(["target", "model", "split"]).reset_index(drop=True)
joint_df.to_csv(ARTIFACT_ROOT / "joint_value_event_diagnostics.csv", index=False)
display(joint_df.head(MAX_DISPLAY_ROWS))


## Section 6 — Scenario quantile-pair evaluation

In [ ]:
pair_rows = []
for (scenario, target), map_row in QUANTILE_PAIR_MAPPING.groupby(["scenario", "target"]):
    q = map_row["selected_quantile"].iloc[0]
    risk_note = map_row["risk_note"].iloc[0]

    for (model, split), g in all_preds.loc[all_preds["target"] == target].groupby(["model", "split"], dropna=False):
        gg = g.dropna(subset=["y_true", q]).copy()
        if gg.empty:
            continue
        y = gg["y_true"].to_numpy(dtype=float)
        yhat = gg[q].to_numpy(dtype=float)
        err = yhat - y
        over = float((err > 0).mean())
        under = float((err < 0).mean())

        # Selected-quantile tail capture proxy.
        tail_true = y >= np.nanquantile(y, 0.9)
        tail_pred = yhat >= np.nanquantile(yhat, 0.9)
        capture = float(np.mean(tail_pred[tail_true])) if tail_true.any() else float("nan")

        pair_rows.append({
            "scenario": scenario,
            "target": target,
            "model": model,
            "split": split,
            "selected_quantile": q,
            "selected_quantile_pinball_loss": float(np.mean(_pinball(y, yhat, int(q[1:]) / 100.0))),
            "selected_quantile_bias": float(np.mean(err)),
            "selected_quantile_hit_rate": float(np.mean(y <= yhat)),
            "overprediction_frequency": over,
            "underprediction_frequency": under,
            "tail_event_capture_selected_quantile": capture,
            "risk_note": risk_note,
        })

pair_df = pd.DataFrame(pair_rows).sort_values(["scenario", "target", "model", "split"]).reset_index(drop=True)
pair_df.to_csv(ARTIFACT_ROOT / "quantile_pair_diagnostics.csv", index=False)
display(pair_df.head(MAX_DISPLAY_ROWS))


## Section 7 — Economic proxy ranking

In [ ]:
weight_rows = []
for target in TARGETS:
    if target == "da_price":
        w = dict(p50_error=0.30, gate=0.25, tail=0.25, calibration=0.15, crossing_penalty=0.05, value_event=0.00)
    elif "capacity" in target:
        w = dict(p50_error=0.15, gate=0.30, tail=0.30, calibration=0.15, crossing_penalty=0.05, value_event=0.05)
    elif "activation_price" in target:
        w = dict(p50_error=0.10, gate=0.20, tail=0.35, calibration=0.15, crossing_penalty=0.05, value_event=0.15)
    else:  # activation rates
        w = dict(p50_error=0.10, gate=0.20, tail=0.30, calibration=0.20, crossing_penalty=0.05, value_event=0.15)
    weight_rows.append({"target": target, **w})
weights = pd.DataFrame(weight_rows)

m = forecast_metrics.merge(gate_metrics, on=["target", "model", "split"], suffixes=("", "_gate"), how="left")

# Aggregate tail and event diagnostics.
tail_agg = tail_df.groupby(["target", "model", "split"], as_index=False).agg(
    tail_score_raw=("tail_mae_over_normal_mae", "mean"),
)
joint_agg = joint_df.groupby(["target", "model", "split"], as_index=False).agg(
    value_event_score_raw=("joint_event_f1", "mean"),
)

score = m.merge(tail_agg, on=["target", "model", "split"], how="left").merge(joint_agg, on=["target", "model", "split"], how="left").merge(weights, on="target", how="left")

# Normalize helper (lower better -> invert after normalization).
def _norm(s: pd.Series, higher_is_better: bool) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    lo, hi = x.min(skipna=True), x.max(skipna=True)
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
        return pd.Series(np.nan, index=s.index)
    z = (x - lo) / (hi - lo)
    return z if higher_is_better else (1.0 - z)

score["p50_error_score"] = _norm(score["mae_p50"], higher_is_better=False)
score["gate_score"] = _norm(score["mae_p50_gate"], higher_is_better=False)
score["tail_score"] = _norm(score["tail_score_raw"], higher_is_better=False)
score["calibration_score"] = _norm((score["coverage_p10_p90"] - 0.8).abs(), higher_is_better=False)
score["crossing_penalty_score"] = _norm(score["crossing_rate_before_repair"], higher_is_better=False)
score["value_event_score"] = _norm(score["value_event_score_raw"], higher_is_better=True)

score["final_composite_score"] = (
    score["p50_error"] * score["p50_error_score"].fillna(0.0)
    + score["gate"] * score["gate_score"].fillna(0.0)
    + score["tail"] * score["tail_score"].fillna(0.0)
    + score["calibration"] * score["calibration_score"].fillna(0.0)
    + score["crossing_penalty"] * score["crossing_penalty_score"].fillna(0.0)
    + score["value_event"] * score["value_event_score"].fillna(0.0)
)

score_out = score[[
    "target", "model", "split",
    "p50_error_score", "gate_score", "tail_score", "calibration_score", "crossing_penalty_score", "value_event_score",
    "final_composite_score",
]].copy().sort_values(["target", "split", "final_composite_score"], ascending=[True, True, False])

weights.to_csv(ARTIFACT_ROOT / "model_selection_score_weights.csv", index=False)
score_out.to_csv(ARTIFACT_ROOT / "model_selection_scores.csv", index=False)
display(score_out.head(MAX_DISPLAY_ROWS))


## Section 8 — Final recommendation table

In [ ]:
reco_rows = []
for (target, split), g in score_out.groupby(["target", "split"], dropna=False):
    best = g.sort_values("final_composite_score", ascending=False).iloc[0]
    reco_rows.append({
        "target": target,
        "split": split,
        "recommended_model": best["model"],
        "runner_model_id": best["model"],
        "reason": "best composite forecast-side score",
        "p50_score": float(best["p50_error_score"]),
        "gate_score": float(best["gate_score"]),
        "tail_score": float(best["tail_score"]),
        "calibration_score": float(best["calibration_score"]),
        "main_weaknesses": "check tail drift and calibration by target",
        "thesis_risk_note": "forecast-side proxy only; not direct pnl guarantee",
        "acceptable_for_simulation": 1,
    })

reco = pd.DataFrame(reco_rows).sort_values(["target", "split"]).reset_index(drop=True)
reco.to_csv(ARTIFACT_ROOT / "final_model_recommendation_table.csv", index=False)

md_lines = [
    "# Final Model Recommendation Table",
    "",
    reco.to_markdown(index=False),
]
(ARTIFACT_ROOT / "final_model_recommendation_table.md").write_text("\n".join(md_lines), encoding="utf-8")

display(reco.head(MAX_DISPLAY_ROWS))


## Section 9 — Optional visual appendix

In [ ]:
if PLOT_ENABLED:
    import matplotlib.pyplot as plt

    # Light reliability-style chart example for one target/model/split.
    sample = forecast_metrics.sort_values("n", ascending=False).head(1)
    if not sample.empty:
        r = sample.iloc[0]
        sub = all_preds[
            (all_preds["target"] == r["target"]) &
            (all_preds["model"] == r["model"]) &
            (all_preds["split"] == r["split"])
        ].dropna(subset=["y_true", "p10", "p30", "p50", "p70", "p90"])

        cov = [
            ((sub["y_true"] >= sub["p10"]) & (sub["y_true"] <= sub["p90"])) .mean(),
            ((sub["y_true"] >= sub["p30"]) & (sub["y_true"] <= sub["p70"])) .mean(),
        ]
        tgt = [0.8, 0.4]

        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(2)
        ax.bar(x - 0.15, cov, width=0.3, label="observed")
        ax.bar(x + 0.15, tgt, width=0.3, label="target")
        ax.set_xticks(x)
        ax.set_xticklabels(["p10-p90", "p30-p70"])
        ax.set_ylim(0, 1)
        ax.legend()
        ax.set_title(f"Coverage check: {r['target']} / {r['model']} / {r['split']}")
        fig.tight_layout()
        fig.savefig(FIG_ROOT / "coverage_example.png", dpi=150)
        plt.close(fig)


## Section 10 — Run manifest

In [ ]:
warnings_list = []
if inventory["missing_required_quantiles"].astype(str).str.len().gt(0).any():
    warnings_list.append("Some inventory rows reported missing required quantiles.")
if inventory["duplicate_rows_snapshot_target_lead"].sum() > 0:
    warnings_list.append("Duplicate snapshot/target/lead rows detected.")

manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_mode": EVALUATION_MODE,
    "model_manifests": {k: str(v) for k, v in MODEL_MANIFESTS.items()},
    "prediction_tables_override": {k: str(v) for k, v in PREDICTION_TABLES.items()},
    "model_ids": MODEL_IDS,
    "splits_evaluated": sorted(all_preds["split"].dropna().astype(str).unique().tolist()),
    "targets_evaluated": TARGETS,
    "quantiles_evaluated": QUANTILES_REQUIRED,
    "key_quantile_pairs": KEY_QUANTILE_PAIRS,
    "outputs": {
        "benchmark_data_inventory": str(ARTIFACT_ROOT / "benchmark_data_inventory.csv"),
        "forecast_metrics_probabilistic": str(ARTIFACT_ROOT / "forecast_metrics_probabilistic.csv"),
        "gate_time_forecast_metrics": str(ARTIFACT_ROOT / "gate_time_forecast_metrics.csv"),
        "tail_performance_value_events": str(ARTIFACT_ROOT / "tail_performance_value_events.csv"),
        "joint_value_event_diagnostics": str(ARTIFACT_ROOT / "joint_value_event_diagnostics.csv"),
        "quantile_pair_diagnostics": str(ARTIFACT_ROOT / "quantile_pair_diagnostics.csv"),
        "quantile_pair_mapping": str(ARTIFACT_ROOT / "quantile_pair_mapping.csv"),
        "model_selection_scores": str(ARTIFACT_ROOT / "model_selection_scores.csv"),
        "model_selection_score_weights": str(ARTIFACT_ROOT / "model_selection_score_weights.csv"),
        "final_model_recommendation_table_csv": str(ARTIFACT_ROOT / "final_model_recommendation_table.csv"),
        "final_model_recommendation_table_md": str(ARTIFACT_ROOT / "final_model_recommendation_table.md"),
    },
    "warnings": warnings_list,
    "missing_data_issues": [],
    "recommended_model_per_target": (
        reco.groupby("target")["recommended_model"].first().to_dict() if not reco.empty else {}
    ),
}

(ARTIFACT_ROOT / "benchmark_notebook_summary.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2)[:2000])
